# 01 — Génération des métriques · Mirabelle

Ce notebook exécute **uniquement** les indicateurs déclarés dans `metric_registry.py`
pour le corpus **Mirabelle**. Les sorties sont isolées dans `csv/Mirabelle/` et les
journaux dans `csv/Mirabelle/logs/`.

Tous les scripts sont lancés dans des sous-processus indépendants afin d'éviter les
effets de bord entre imports, loggers et arguments de ligne de commande.


## 1. Configuration


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from metric_registry import get_dataset
from pipeline_utils import detect_project_dir, inspect_input, run_metric_jobs, metric_inventory

PROJECT_DIR = detect_project_dir()
DATASET = "Mirabelle"
SPEC = get_dataset(DATASET)

# Laisser à None pour utiliser le chemin défini dans metric_registry.py.
INPUT_OVERRIDE = None

OVERWRITE_OUTPUTS = True

print("Projet       :", PROJECT_DIR)
print("Corpus       :", DATASET)
print("Scripts      :", SPEC.scripts_path(PROJECT_DIR))
print("Sorties CSV  :", SPEC.csv_dir(PROJECT_DIR))
print("Entrée défaut:", SPEC.input(PROJECT_DIR))


Projet       : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine nettoyée\Chaine_nettoyee
Corpus       : Mirabelle
Scripts      : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine nettoyée\Chaine_nettoyee\scripts_Mirabelle
Sorties CSV  : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine nettoyée\Chaine_nettoyee\csv\Mirabelle
Entrée défaut: C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine nettoyée\Chaine_nettoyee\data\donnees_2024-09-20_10h15-11h45.csv


## 2. Contrôle de l'entrée


In [2]:
input_df, input_summary = inspect_input(PROJECT_DIR, SPEC, INPUT_OVERRIDE)
display(input_summary)
print("Colonnes disponibles :")
print(", ".join(input_df.columns.astype(str)))


,corpus,fichier,lignes,colonnes,sujets_uniques,colonne_identifiant
0,Mirabelle,donnees_2024-09-20_10h15-11h45.csv,3487,34,49,actor


Colonnes disponibles :
research_usage, result, _id.$oid, timestamp.$date, stored.$date, actor, binome, actor.objectType, verb, object, object.objectType, object.extension, session.id, commandRan, result.success, stdin, stdout, stderr, P_codeState, filename, filename_infere, F_codeState, Debug_TimeStampEnd, Debug_TimeStampActions, tests, function, time_delta, session.duration, seance, TP, Type_TP, research_usage_clean, binome_is_debutant, actor_is_debutant


## 3. Indicateurs déclarés


In [3]:
jobs_df = pd.DataFrame([
    {
        "script": item.script,
        "sortie": item.output,
        "métrique": item.metric,
        "description": item.description,
        "colonnes requises": ", ".join(item.required_columns),
        "arguments": " ".join(item.extra_args),
    }
    for item in SPEC.jobs
])
display(jobs_df)


,script,sortie,métrique,description,colonnes requises,arguments
0,session_count_Mirabelle.py,session_count.csv,SessionCount,Nombre de sessions d'activité séparées par plu...,"actor, verb, timestamp.$date",5.0
1,session_span_Mirabelle.py,session_span.csv,SessionSpanMinutes,Durée moyenne des sessions Run.Test séparées p...,"actor, verb, timestamp.$date",
2,test_pass_rate_Mirabelle.py,test_pass_rate.csv,TestPassRate,Proportion globale des cas de test étudiants q...,"actor, verb, tests",
3,failed_run_ratio_Mirabelle.py,failed_run_ratio.csv,FailedTestRunRatio,Part des Run.Test non vides contenant au moins...,"actor, verb, tests",
4,exception_run_ratio_Mirabelle.py,exception_run_ratio.csv,ExceptionTestRunRatio,Part des Run.Test non vides contenant au moins...,"actor, verb, tests",
5,function_coverage_Mirabelle.py,function_coverage.csv,FunctionCoverage,Nombre de fonctions considérées comme traitées.,"actor, verb, tests, P_codeState",
6,eq_FE_Mirabelle.py,error_quotient_fe.csv,ErrorQuotient_FE,"Error Quotient, identité d'erreur au niveau du...","actor, verb, tests, timestamp.$date",
7,eq_Mirabelle.py,error_quotient.csv,ErrorQuotient,"Error Quotient, identité d'erreur au niveau du...","actor, verb, tests, timestamp.$date",
8,red_FE_Mirabelle.py,red_fe.csv,RED_FE,"Repeated Error Density, identité d'erreur au n...","actor, verb, tests, timestamp.$date",
9,red_Mirabelle.py,red.csv,RED,"Repeated Error Density, identité d'erreur au n...","actor, verb, tests, timestamp.$date",


## 4. Exécution


In [4]:
report_df = run_metric_jobs(
    PROJECT_DIR,
    DATASET,
    overwrite=OVERWRITE_OUTPUTS,
    input_override=INPUT_OVERRIDE,
)
display(report_df[[
    "script", "metric", "status", "rows", "columns", "message", "log_path"
]])

errors = report_df[~report_df["status"].isin(["ok", "skipped_existing"])]
if errors.empty:
    print("Tous les indicateurs ont été générés correctement.")
else:
    print(f"{len(errors)} indicateur(s) à vérifier. Consultez les fichiers log_path.")


,script,metric,status,rows,columns,message,log_path
0,session_count_Mirabelle.py,SessionCount,ok,47,"SubjectID, SessionCount",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
1,session_span_Mirabelle.py,SessionSpanMinutes,ok,47,"SubjectID, SessionSpanMinutes",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
2,test_pass_rate_Mirabelle.py,TestPassRate,ok,47,"SubjectID, TestPassRate",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
3,failed_run_ratio_Mirabelle.py,FailedTestRunRatio,ok,47,"SubjectID, FailedTestRunRatio",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
4,exception_run_ratio_Mirabelle.py,ExceptionTestRunRatio,ok,47,"SubjectID, ExceptionTestRunRatio",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
5,function_coverage_Mirabelle.py,FunctionCoverage,ok,49,"SubjectID, FunctionCoverage",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
6,eq_FE_Mirabelle.py,ErrorQuotient_FE,ok,39,"SubjectID, ErrorQuotient_FE",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
7,eq_Mirabelle.py,ErrorQuotient,ok,39,"SubjectID, ErrorQuotient",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
8,red_FE_Mirabelle.py,RED_FE,ok,39,"SubjectID, RED_FE",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
9,red_Mirabelle.py,RED,ok,39,"SubjectID, RED",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...


Tous les indicateurs ont été générés correctement.


## 5. Inventaire des sorties


In [5]:
inventory_df = metric_inventory(SPEC.csv_dir(PROJECT_DIR))
display(inventory_df)


,fichier,statut,lignes,colonnes,variables
0,attempts_to_first_success.csv,ok,46,2,AttemptsToFirstSuccess
1,code_change_magnitude.csv,ok,38,2,CodeChangeMagnitude
2,error_quotient.csv,ok,39,2,ErrorQuotient
3,error_quotient_fe.csv,ok,39,2,ErrorQuotient_FE
4,exception_run_ratio.csv,ok,47,2,ExceptionTestRunRatio
5,failed_run_ratio.csv,ok,47,2,FailedTestRunRatio
6,first_attempt_success_rate.csv,ok,47,2,FirstAttemptSuccessRate
7,function_coverage.csv,ok,49,2,FunctionCoverage
8,productive_transition_rate.csv,ok,37,2,ProductiveTransitionRate
9,red.csv,ok,39,2,RED
